In [1]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from tokenizers import decoders

import numpy as np
from sklearn.model_selection import train_test_split

from torchtune.modules import RotaryPositionalEmbeddings
from torch.nn import Transformer
import matplotlib.pyplot as plt
from tqdm import tqdm
import sacrebleu
from timeit import default_timer as timer
from datasets import load_dataset

%matplotlib inline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
import os

os.makedirs('data', exist_ok=True)


In [4]:
def download_file(url, filename):
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        !wget -q {url} -O {filename}

In [5]:
BASE_URL = "https://data.statmt.org/opus-100-corpus/v1.0/supervised"

# Языковые пары: исходный язык -> целевой (en)
LANG_PAIRS = {
    'cs': 'Czech',
    'de': 'German',
    'da': 'Danish'
}

In [6]:
MAX_TRAIN_LINES = 50000

all_train_src = []
all_train_tgt = []

for lang, lang_name in LANG_PAIRS.items():
    print(f"\nЗагрузка {lang_name} -> English...")

    src_url = f"{BASE_URL}/{lang}-en/opus.{lang}-en-train.{lang}"
    tgt_url = f"{BASE_URL}/{lang}-en/opus.{lang}-en-train.en"

    src_file = f'data/opus.{lang}-en-train.{lang}'
    tgt_file = f'data/opus.{lang}-en-train.en'

    download_file(src_url, src_file)
    download_file(tgt_url, tgt_file)

    with open(src_file, 'r', encoding='utf-8') as f:
        src_lines = f.read().splitlines()
    with open(tgt_file, 'r', encoding='utf-8') as f:
        tgt_lines = f.read().splitlines()

    if MAX_TRAIN_LINES:
        src_lines = src_lines[:MAX_TRAIN_LINES]
        tgt_lines = tgt_lines[:MAX_TRAIN_LINES]

    lang_prefix = f"__{lang}__"
    src_lines = [f"{lang_prefix} {line}" for line in src_lines]

    all_train_src.extend(src_lines)
    all_train_tgt.extend(tgt_lines)

    print(f"  {lang_name}: {len(src_lines)} примеров")

print(f"\nВсего обучающих примеров: {len(all_train_src)}")


Загрузка Czech -> English...
  Czech: 50000 примеров

Загрузка German -> English...
  German: 50000 примеров

Загрузка Danish -> English...
  Danish: 50000 примеров

Всего обучающих примеров: 150000


In [7]:
test_data = {}

for lang, lang_name in LANG_PAIRS.items():
    print(f"Загрузка теста {lang_name} -> English...")

    src_url = f"{BASE_URL}/{lang}-en/opus.{lang}-en-test.{lang}"
    tgt_url = f"{BASE_URL}/{lang}-en/opus.{lang}-en-test.en"

    src_file = f'data/opus.{lang}-en-test.{lang}'
    tgt_file = f'data/opus.{lang}-en-test.en'

    download_file(src_url, src_file)
    download_file(tgt_url, tgt_file)

    with open(src_file, 'r', encoding='utf-8') as f:
        src_lines = f.read().splitlines()
    with open(tgt_file, 'r', encoding='utf-8') as f:
        tgt_lines = f.read().splitlines()

    lang_prefix = f"__{lang}__"
    src_lines = [f"{lang_prefix} {line}" for line in src_lines]

    test_data[lang] = {
        'src': src_lines,
        'tgt': tgt_lines,
        'name': lang_name
    }

    print(f"  {lang_name} test: {len(src_lines)} примеров")

Загрузка теста Czech -> English...
  Czech test: 2000 примеров
Загрузка теста German -> English...
  German test: 2000 примеров
Загрузка теста Danish -> English...
  Danish test: 2000 примеров


In [8]:
with open('data/all_train_src.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(all_train_src))
with open('data/all_train_tgt.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(all_train_tgt))

In [9]:
tokenizer_src = Tokenizer(BPE())
tokenizer_src.pre_tokenizer = Whitespace()

special_tokens_src = ["[PAD]", "[BOS]", "[EOS]", "[UNK]", "__cs__", "__de__", "__da__"]

trainer_src = BpeTrainer(
    special_tokens=special_tokens_src,
    vocab_size=30000,
    min_frequency=2,
    end_of_word_suffix=' '
)
tokenizer_src.train(files=['data/all_train_src.txt'], trainer=trainer_src)
tokenizer_src.decoder = decoders.BPEDecoder()
tokenizer_src.save('data/tokenizer_src')

In [10]:
tokenizer_tgt = Tokenizer(BPE())
tokenizer_tgt.pre_tokenizer = Whitespace()
special_tokens_tgt = ["[PAD]", "[BOS]", "[EOS]", "[UNK]"]

trainer_tgt = BpeTrainer(
    special_tokens=special_tokens_tgt,
    vocab_size=30000,
    min_frequency=2,
    end_of_word_suffix=' '
)
tokenizer_tgt.train(files=['data/all_train_tgt.txt'], trainer=trainer_tgt)
tokenizer_tgt.decoder = decoders.BPEDecoder()
tokenizer_tgt.save('data/tokenizer_tgt')

In [11]:
tokenizer_src = Tokenizer.from_file('data/tokenizer_src')
tokenizer_tgt = Tokenizer.from_file('data/tokenizer_tgt')

print(f"Vocab src: {tokenizer_src.get_vocab_size()}, tgt: {tokenizer_tgt.get_vocab_size()}")

Vocab src: 30000, tgt: 30000


In [13]:
PAD_IDX = tokenizer_tgt.token_to_id('[PAD]')
BOS_IDX = tokenizer_tgt.token_to_id('[BOS]')
EOS_IDX = tokenizer_tgt.token_to_id('[EOS]')

max_len_src = 64
max_len_tgt = 64

def encode_src(text, tokenizer, max_len):
    return tokenizer.encode(text).ids[:max_len]

def encode_tgt(text, tokenizer, max_len):
    encoded = tokenizer.encode(text).ids[:max_len-2]
    return [BOS_IDX] + encoded + [EOS_IDX]

X_src_train = [encode_src(t, tokenizer_src, max_len_src) for t in all_train_src]
X_tgt_train = [encode_tgt(t, tokenizer_tgt, max_len_tgt) for t in all_train_tgt]

X_src_train, X_src_valid, X_tgt_train, X_tgt_valid = train_test_split(
    X_src_train, X_tgt_train, test_size=0.05, random_state=42
)

def prepare_test(src_list, tgt_list):
    return ([encode_src(t, tokenizer_src, max_len_src) for t in src_list],
            [encode_tgt(t, tokenizer_tgt, max_len_tgt) for t in tgt_list])

test_encoded = {}
for lang, data in test_data.items():
    test_encoded[lang] = prepare_test(data['src'], data['tgt'])

print(f"Train: {len(X_src_train)}, Valid: {len(X_src_valid)}")
for lang, (src, tgt) in test_encoded.items():
    print(f"Test {lang}: {len(src)} примеров")

Train: 142500, Valid: 7500
Test cs: 2000 примеров
Test de: 2000 примеров
Test da: 2000 примеров


In [14]:
class MultilingualDataset(torch.utils.data.Dataset):
    def __init__(self, texts_src, texts_tgt):
        self.texts_src = [torch.LongTensor(sent) for sent in texts_src]
        self.texts_src = torch.nn.utils.rnn.pad_sequence(
            self.texts_src, batch_first=True, padding_value=PAD_IDX
        )
        self.texts_tgt = [torch.LongTensor(sent) for sent in texts_tgt]
        self.texts_tgt = torch.nn.utils.rnn.pad_sequence(
            self.texts_tgt, batch_first=True, padding_value=PAD_IDX
        )
        self.length = len(texts_src)

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        return self.texts_src[index], self.texts_tgt[index]

batch_size = 64
training_set = MultilingualDataset(X_src_train, X_tgt_train)
training_generator = torch.utils.data.DataLoader(training_set, batch_size=batch_size, shuffle=True, drop_last=True)
valid_set = MultilingualDataset(X_src_valid, X_tgt_valid)
valid_generator = torch.utils.data.DataLoader(valid_set, batch_size=batch_size, shuffle=False)

In [15]:
def make_sliding_window_mask_with_sink(seq_len, window_size=128):
    half_window = window_size // 2
    positions = torch.arange(seq_len)
    dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    mask = dist > half_window
    mask[:, 0] = False
    return mask

def make_causal_sliding_window_mask_with_sink(seq_len, window_size=128):
    half_window = window_size // 2
    positions = torch.arange(seq_len)
    dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    causal = ~torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
    local = dist > half_window
    mask = causal | local
    mask[:, 0] = False
    return mask

def get_layer_types(num_layers, global_every_n=3):
    return ["global" if i % global_every_n == 0 or i == (num_layers-1) else "local" for i in range(num_layers)]


In [16]:
class EncoderLayerAlternating(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout, attention_type, window_size=128):
        super().__init__()
        self.attention_type = attention_type
        self.window_size = window_size
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None):
        S = src.size(1)
        attn_mask = make_sliding_window_mask_with_sink(S, self.window_size).to(src.device) if self.attention_type == "local" else None
        src2 = self.norm1(src)
        src2, _ = self.self_attn(src2, src2, src2, attn_mask=attn_mask, key_padding_mask=src_key_padding_mask)
        src2 = torch.nan_to_num(src2)
        src = src + self.dropout(src2)
        src2 = self.norm2(src)
        src2 = self.ff(src2)
        return src + self.dropout(src2)

In [17]:
class DecoderLayerAlternating(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout, attention_type, window_size=128):
        super().__init__()
        self.attention_type = attention_type
        self.window_size = window_size
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        T = tgt.size(1)
        if self.attention_type == "local":
            tgt_mask = make_causal_sliding_window_mask_with_sink(T, self.window_size).to(tgt.device)
        else:
            tgt_mask = (~torch.tril(torch.ones((T, T), dtype=torch.bool))).to(tgt.device)
        tgt2 = self.norm1(tgt)
        tgt2, _ = self.self_attn(tgt2, tgt2, tgt2, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        tgt2 = torch.nan_to_num(tgt2)
        tgt = tgt + self.dropout(tgt2)
        tgt2 = self.norm2(tgt)
        tgt2, _ = self.cross_attn(tgt2, memory, memory, key_padding_mask=memory_key_padding_mask)
        tgt = tgt + self.dropout(tgt2)
        tgt2 = self.norm3(tgt)
        tgt2 = self.ff(tgt2)
        return tgt + self.dropout(tgt2)

In [18]:
class MultilingualTransformer(nn.Module):
    def __init__(self, vocab_size_enc, vocab_size_dec, d_model, num_heads, ff_dim, num_layers, dropout=0.1, global_every_n=3, window_size=128):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.embedding_enc = nn.Embedding(vocab_size_enc, d_model)
        self.embedding_dec = nn.Embedding(vocab_size_dec, d_model)
        self.positional_encoding = RotaryPositionalEmbeddings(d_model // num_heads)
        layer_types = get_layer_types(num_layers, global_every_n)
        self.encoder_layers = nn.ModuleList([EncoderLayerAlternating(d_model, num_heads, ff_dim, dropout, lt, window_size) for lt in layer_types])
        self.decoder_layers = nn.ModuleList([DecoderLayerAlternating(d_model, num_heads, ff_dim, dropout, lt, window_size) for lt in layer_types])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size_dec, bias=False)
        self.output_layer.weight = self.embedding_dec.weight  # weight tying
        self.layer_types = layer_types

    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None):
        src_embedded = self.embedding_enc(src)
        B, S, E = src_embedded.shape
        src_embedded = self.positional_encoding(src_embedded.view(B, S, self.num_heads, E // self.num_heads)).view(B, S, E)
        tgt_embedded = self.embedding_dec(tgt)
        B, T, E = tgt_embedded.shape
        tgt_embedded = self.positional_encoding(tgt_embedded.view(B, T, self.num_heads, E // self.num_heads)).view(B, T, E)
        memory = src_embedded
        for layer in self.encoder_layers:
            memory = layer(memory, src_key_padding_mask=src_key_padding_mask)
        output = tgt_embedded
        for layer in self.decoder_layers:
            output = layer(output, memory, tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=src_key_padding_mask)
        output = self.final_norm(output)
        return self.output_layer(output)

In [19]:
def train_epoch(model, iterator, optimizer, loss_fn, scheduler):
    epoch_loss = []
    model.train()
    pbar = tqdm(iterator, desc='Train', leave=False)
    for texts_src, texts_tgt in pbar:
        texts_src, texts_tgt = texts_src.to(DEVICE), texts_tgt.to(DEVICE)
        texts_tgt_input, texts_tgt_out = texts_tgt[:, :-1], texts_tgt[:, 1:]
        src_padding_mask = (texts_src == PAD_IDX)
        tgt_padding_mask = (texts_tgt_input == PAD_IDX)
        optimizer.zero_grad()
        logits = model(texts_src, texts_tgt_input, src_padding_mask, tgt_padding_mask)
        B, S, C = logits.shape
        loss = loss_fn(logits.reshape(B * S, C), texts_tgt_out.reshape(B * S))
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss.append(loss.item())
        pbar.set_postfix(loss=f'{np.mean(epoch_loss):.4f}')
    return np.mean(epoch_loss)

In [20]:
@torch.no_grad()
def evaluate(model, iterator, loss_fn):
    epoch_loss = []
    model.eval()
    for texts_src, texts_tgt in tqdm(iterator, desc='Eval', leave=False):
        texts_src, texts_tgt = texts_src.to(DEVICE), texts_tgt.to(DEVICE)
        texts_tgt_input, texts_tgt_out = texts_tgt[:, :-1], texts_tgt[:, 1:]
        src_padding_mask = (texts_src == PAD_IDX)
        tgt_padding_mask = (texts_tgt_input == PAD_IDX)
        logits = model(texts_src, texts_tgt_input, src_padding_mask, tgt_padding_mask)
        B, S, C = logits.shape
        loss = loss_fn(logits.reshape(B * S, C), texts_tgt_out.reshape(B * S))
        epoch_loss.append(loss.item())
    return np.mean(epoch_loss)

In [21]:
@torch.no_grad()
def translate_batch(texts, model, max_len=64):
    model.eval()
    batch_size = len(texts)

    input_ids_list = [tokenizer_src.encode(text).ids[:max_len_src] for text in texts]
    max_src_len = max(len(ids) for ids in input_ids_list)
    input_ids_padded = [ids + [PAD_IDX] * (max_src_len - len(ids)) for ids in input_ids_list]
    input_ids_tensor = torch.LongTensor(input_ids_padded).to(DEVICE)
    src_padding_mask = (input_ids_tensor == PAD_IDX)

    output_ids = [[BOS_IDX] for _ in range(batch_size)]
    eos_id = EOS_IDX
    active_mask = [True] * batch_size

    for step in range(max_len):
        max_out_len = max(len(ids) for i, ids in enumerate(output_ids) if active_mask[i])
        output_ids_padded = []
        for i, ids in enumerate(output_ids):
            padded = ids + [PAD_IDX] * (max_out_len - len(ids))
            output_ids_padded.append(padded)

        output_ids_tensor = torch.LongTensor(output_ids_padded).to(DEVICE)
        tgt_padding_mask = (output_ids_tensor == PAD_IDX)

        logits = model(input_ids_tensor, output_ids_tensor, src_padding_mask, tgt_padding_mask)

        for i in range(batch_size):
            if active_mask[i]:
                pred = logits[i, -1].argmax().item()
                if pred in (eos_id, PAD_IDX):
                    active_mask[i] = False
                    output_ids[i].append(eos_id)
                else:
                    output_ids[i].append(pred)

        if not any(active_mask):
            break

    results = []
    for ids in output_ids:
        tokens = [tokenizer_tgt.id_to_token(i) for i in ids[1:] if i not in (BOS_IDX, EOS_IDX, PAD_IDX)]
        results.append(tokenizer_tgt.decoder.decode(tokens))
    return results

In [26]:
vocab_size_enc = tokenizer_src.get_vocab_size()
vocab_size_dec = tokenizer_tgt.get_vocab_size()
d_model = 256
num_heads = 8
ff_dim = d_model * 4
num_layers = 8
window_size = 32

model = MultilingualTransformer(vocab_size_enc, vocab_size_dec, d_model, num_heads, ff_dim, num_layers, window_size=window_size)
model = model.to(DEVICE)

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=0.0001)
NUM_EPOCHS = 10
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001, pct_start=0.05, steps_per_epoch=len(training_generator), epochs=NUM_EPOCHS)

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Layer types: {model.layer_types}")

Parameters: 30.11M
Layer types: ['global', 'local', 'local', 'global', 'local', 'local', 'global', 'global']


In [27]:
losses = []
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = timer()
    train_loss = train_epoch(model, training_generator, optimizer, loss_fn, scheduler)
    end_time = timer()
    val_loss = evaluate(model, valid_generator, loss_fn)

    if val_loss < best_val_loss:
        print(f'Improved from {best_val_loss:.4f} to {val_loss:.4f}, saving model..')
        torch.save(model.state_dict(), 'model_multilingual.pt')
        best_val_loss = val_loss

    losses.append(val_loss)

    test_texts = [test_data[lang]['src'][1] for lang in ['cs', 'de', 'da']]
    translations = translate_batch(test_texts, model)

    print(f"Epoch {epoch}: Train={train_loss:.3f}, Val={val_loss:.3f}, Time={end_time-start_time:.1f}s")
    for txt, trans in zip(test_texts, translations):
        print(f"  {txt[:50]}... -> {trans[:50]}...")
    print()

torch.save(model.state_dict(), 'model_multilingual_final.pt')
print("Обучение завершено!")

Improved from inf to 6.4880, saving model..
Epoch 1: Train=12.039, Val=6.488, Time=549.0s
  __cs__ - věrného a vzrušeného.... -> - I ' m not a and a and a and a and a and the and ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The the European European European European Europe...
  __da__ Det bliver... mit mesterværk.... -> It ' s a ... ...



Improved from 6.4880 to 6.0700, saving model..
Epoch 2: Train=6.354, Val=6.070, Time=548.1s
  __cs__ - věrného a vzrušeného.... -> - I ' ll be a little on the time . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The the the the the the the the the the the the th...
  __da__ Det bliver... mit mesterværk.... -> It ' s my life . ...



Improved from 6.0700 to 5.7764, saving model..
Epoch 3: Train=5.958, Val=5.776, Time=546.9s
  __cs__ - věrného a vzrušeného.... -> - The world and the world . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The first in the first - in the first - in the fir...
  __da__ Det bliver... mit mesterværk.... -> It ' s my father . ...



Improved from 5.7764 to 5.5559, saving model..
Epoch 4: Train=5.656, Val=5.556, Time=546.9s
  __cs__ - věrného a vzrušeného.... -> - I ' m gonna be a good and a good . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The first of the first year in the first year . ...
  __da__ Det bliver... mit mesterværk.... -> It ' s my own . ...



Improved from 5.5559 to 5.3850, saving model..
Epoch 5: Train=5.404, Val=5.385, Time=546.4s
  __cs__ - věrného a vzrušeného.... -> - They ' re gonna be a little . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The new of the new of the new year . ...
  __da__ Det bliver... mit mesterværk.... -> It ' s gonna be my own . ...



Improved from 5.3850 to 5.2687, saving model..
Epoch 6: Train=5.181, Val=5.269, Time=546.4s
  __cs__ - věrného a vzrušeného.... -> - They ' re not gonna be a few days . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The first of the first year . ...
  __da__ Det bliver... mit mesterværk.... -> It ' s gonna be my way . ...



Improved from 5.2687 to 5.2215, saving model..
Epoch 7: Train=4.977, Val=5.221, Time=547.0s
  __cs__ - věrného a vzrušeného.... -> - A little girl and be a little . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The number of the number of the number of the numb...
  __da__ Det bliver... mit mesterværk.... -> It ' s gonna be my dear . ...



Improved from 5.2215 to 5.2108, saving model..
Epoch 8: Train=4.795, Val=5.211, Time=546.8s
  __cs__ - věrného a vzrušeného.... -> - A little girl and be . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The first of the 1 . ...
  __da__ Det bliver... mit mesterværk.... -> It ' s gonna be my dear . ...



Epoch 9: Train=4.659, Val=5.229, Time=547.2s
  __cs__ - věrného a vzrušeného.... -> - A little girl and be crazy . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The most of the number of the whole year . ...
  __da__ Det bliver... mit mesterværk.... -> It ' s gonna be my dear . ...



Epoch 10: Train=4.593, Val=5.244, Time=548.4s
  __cs__ - věrného a vzrušeného.... -> - A little girl and be crazy . ...
  __de__ Prähistorische Archäologie im Dritten Reich... -> The amount of the number of the amount of the amou...
  __da__ Det bliver... mit mesterværk.... -> It ' s gonna be my dear . ...

Обучение завершено!


In [28]:
model.eval()

def evaluate_language(src_data, tgt_data, lang_name, n_samples=500):
    """Оценивает качество перевода на тестовом наборе"""
    hypotheses = translate_batch(src_data[:n_samples], model)
    references = tgt_data[:n_samples]

    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])

    print(f"\n{'='*50}")
    print(f"Language: {lang_name} | Samples: {len(hypotheses)}")
    print(f"BLEU: {bleu.score:.2f} | chrF2: {chrf.score:.2f}")
    print(f"{'='*50}")

    for i in range(min(3, len(hypotheses))):
        print(f"\n{i+1}. SRC: {src_data[i]}")
        print(f"   REF: {references[i]}")
        print(f"   HYP: {hypotheses[i]}")

    return hypotheses, references, bleu.score, chrf.score

print("Оценка качества перевода...")

results = {}
for lang, data in test_data.items():
    hyp, ref, bleu, chrf = evaluate_language(data['src'], data['tgt'], data['name'], 500)
    results[lang] = {'hyp': hyp, 'ref': ref, 'bleu': bleu, 'chrf': chrf}


print(f"{'Language':<15} {'BLEU':<15} {'chrF2':<15}")
print("-"*60)
for lang, data in test_data.items():
    print(f"{data['name']:<15} {results[lang]['bleu']:<15.2f} {results[lang]['chrf']:<15.2f}")
print("="*60)

Оценка качества перевода...

Language: Czech | Samples: 500
BLEU: 9.01 | chrF2: 22.72

1. SRC: __cs__ - Drž hubu!
   REF: - Fuck you,
   HYP: - Shut up ! 

2. SRC: __cs__ - věrného a vzrušeného.
   REF: Loyal, and aroused.
   HYP: - A little girl and be crazy . 

3. SRC: __cs__ on již stane se v korporacním precinu století.
   REF: It has evolved to the corporate crime of the century.
   HYP: The most of the month on the German of the DD . 

Language: German | Samples: 500
BLEU: 7.94 | chrF2: 21.15

1. SRC: __de__ 04:26:35
   REF: 04:26:35
   HYP: 04 : 26 : 26 

2. SRC: __de__ Prähistorische Archäologie im Dritten Reich".
   REF: Prähistorische Archäologie im Dritten Reich".
   HYP: The amount of the number of the amount of the amount of the new year . 

3. SRC: __de__ Die Nutzungsbedingungen werden durch das Klicken des Nutzers auf "Profil Speichern" vereinbart.
   REF: By clicking on 'Save profile', you the user agree to these terms and conditions.
   HYP: The Internet of the same sy

In [30]:
def find_best_translations(hypotheses, references, src_texts, n_best=4):
    """Находит лучшие переводы по предложению BLEU"""
    scores = [sacrebleu.sentence_bleu(hyp, [ref]).score for hyp, ref in zip(hypotheses, references)]
    best_idx = np.argsort(scores)[-n_best:][::-1]

    print(f"\nЛучшие {n_best} переводов:")
    for idx in best_idx:
        print(f"\nBLEU: {scores[idx]:.2f}")
        print(f"SRC: {src_texts[idx]}")
        print(f"REF: {references[idx]}")
        print(f"HYP: {hypotheses[idx]}")
    return best_idx, scores


for lang, data in test_data.items():
    print(f"\n--- {data['name']} ---")
    find_best_translations(results[lang]['hyp'], results[lang]['ref'], data['src'], n_best=15)


--- Czech ---

Лучшие 15 переводов:

BLEU: 100.00
SRC: __cs__ Ne, Leo.
REF: No, Leo.
HYP: No , Leo . 

BLEU: 100.00
SRC: __cs__ A Peter.
REF: And Peter.
HYP: And Peter . 

BLEU: 100.00
SRC: __cs__ To bylo dobré.
REF: That was good.
HYP: That was good . 

BLEU: 100.00
SRC: __cs__ - Ano.
REF: - Yes.
HYP: - Yes . 

BLEU: 100.00
SRC: __cs__ Ano.
REF: Yes.
HYP: Yes . 

BLEU: 100.00
SRC: __cs__ 40-15.
REF: 40-15.
HYP: 40 - 15 . 

BLEU: 100.00
SRC: __cs__ Ano.
REF: Yes.
HYP: Yes . 

BLEU: 100.00
SRC: __cs__ Harina?
REF: Harina?
HYP: Harina ? 

BLEU: 100.00
SRC: __cs__ Prostě se jich zbavte.
REF: Just get rid of them.
HYP: Just get rid of them . 

BLEU: 72.52
SRC: __cs__ Pagina cerută: http://agenda.liternet.ro/vot/db.php?j=4&q=7823&t=articole&c=5
REF: Pagina cerută: http://agenda.liternet.ro/vot/db.php?j=3&q=11001&t=articole&c=5
HYP: Pagina cerută : http :// agenda . liternet . ro / vot / db . php ? j = 4 & q = articole & t = 5 

BLEU: 71.03
SRC: __cs__ Schwedenplatz, 1010 Wien Points of int

In [48]:

cy_src_url = f"{BASE_URL}/cy-en/opus.cy-en-test.cy"
cy_tgt_url = f"{BASE_URL}/cy-en/opus.cy-en-test.en"
cy_src_file = 'data/opus.cy-en-test.cy'
cy_tgt_file = 'data/opus.cy-en-test.en'

download_file(cy_src_url, cy_src_file)
download_file(cy_tgt_url, cy_tgt_file)

if os.path.exists(cy_src_file) and os.path.exists(cy_tgt_file):
    with open(cy_src_file, 'r', encoding='utf-8') as f:
        cy_test_src = [line.strip() for line in f.read().splitlines() if line.strip()]
    with open(cy_tgt_file, 'r', encoding='utf-8') as f:
        cy_test_tgt = [line.strip() for line in f.read().splitlines() if line.strip()]

    if len(cy_test_src) > 0:
        print(f"Загружено {len(cy_test_src)} примеров валлийского языка")

        chosen_prefix = "__de__"
        cy_test_src_prefixed = [f"{chosen_prefix} {s}" for s in cy_test_src]

        print(f"Используем префикс {chosen_prefix} для имитации известного языка...")

        cy_hypotheses = translate_batch(cy_test_src_prefixed[:100], model)

        print("\nПримеры zero-shot переводов (CY->EN):")
        for i in range(min(5, len(cy_hypotheses))):
            print(f"\n{i+1}. CY:  {cy_test_src[i]}")
            print(f"   REF: {cy_test_tgt[i]}")
            print(f"   HYP: {cy_hypotheses[i]}")

        if len(cy_hypotheses) > 0:
            cy_bleu = sacrebleu.corpus_bleu(cy_hypotheses, [cy_test_tgt])
            cy_chrf = sacrebleu.corpus_chrf(cy_hypotheses, [cy_test_tgt])

            print(f"\n{'='*50}")
            print(f"Zero-shot Welsh (via {chosen_prefix})")
            print(f"BLEU: {cy_bleu.score:.2f}")
            print(f"chrF2: {cy_chrf.score:.2f}")
            print(f"{'='*50}")
        else:
            print("Не удалось сгенерировать переводы")


Загружено 2000 примеров валлийского языка
Используем префикс __de__ для имитации известного языка...

Примеры zero-shot переводов (CY->EN):

1. CY:  Llyfrnodau
   REF: No bookmarks
   HYP: LLLccccci 

2. CY:  Cyfansoddi ateb at bob un o dderbynwyr y neges a ddewiswyd
   REF: Compose a reply to all the recipients of the selected message
   HYP: Fb ' s his name is a a big - a - a - a - a - a - a - a - a - a - a - a - - - - - - - - jb . 

3. CY:  5 o'r un Fath [50]
   REF: 5 of a Kind [50]
   HYP: 5 ' Fh [ h ] 

4. CY:  _Arddull ateb:
   REF: Memo layout style
   HYP: Ardddddddddddddd: 

5. CY:  'Dych chi wedi ennill!
   REF: You win!
   HYP: ' Cause I ' m doing it ! 

Zero-shot Welsh (via __de__)
BLEU: 0.73
chrF2: 3.82


In [46]:
def evaluate_language_full(src_data, tgt_data, lang_name, batch_size=20):
    """
    Оценивает качество перевода на ВСЁМ тестовом наборе.
    Обрабатывает данные батчами для эффективности.
    """
    all_hypotheses = []
    all_references = []

    n_total = len(src_data)
    n_batches = (n_total + batch_size - 1) // batch_size  # ceiling division

    print(f"Оценка языка: {lang_name}")
    print(f"Всего примеров: {n_total}, Батчей: {n_batches}")


    # Обрабатываем все данные батчами
    for i in tqdm(range(0, n_total, batch_size), desc=f'{lang_name}'):
        batch_src = src_data[i:i+batch_size]
        batch_tgt = tgt_data[i:i+batch_size]

        # Генерируем переводы для батча
        batch_hyp = translate_batch(batch_src, model)

        all_hypotheses.extend(batch_hyp)
        all_references.extend(batch_tgt)

    # Считаем метрики на всех данных
    bleu = sacrebleu.corpus_bleu(all_hypotheses, [all_references])
    chrf = sacrebleu.corpus_chrf(all_hypotheses, [all_references])

    print(f"Language: {lang_name} | Total Samples: {len(all_hypotheses)}")
    print(f"BLEU: {bleu.score:.2f} | chrF2: {chrf.score:.2f}")


    # Показываем первые 3 примера
    print("\nПримеры переводов:")
    for i in range(min(3, len(all_hypotheses))):
        print(f"\n{i+1}. SRC: {src_data[i]}")
        print(f"   REF: {all_references[i]}")
        print(f"   HYP: {all_hypotheses[i]}")

    return all_hypotheses, all_references, bleu.score, chrf.score


results = {}
for lang, data in test_data.items():
    hyp, ref, bleu, chrf = evaluate_language_full(
        data['src'],
        data['tgt'],
        data['name'],
        batch_size=20
    )
    results[lang] = {
        'hyp': hyp,
        'ref': ref,
        'bleu': bleu,
        'chrf': chrf,
        'n_samples': len(hyp)
    }


print(f"{'Language':<15} {'Samples':<12} {'BLEU':<15} {'chrF2':<15}")
print("-"*70)
for lang, data in test_data.items():
    n = results[lang]['n_samples']
    bleu = results[lang]['bleu']
    chrf = results[lang]['chrf']
    print(f"{data['name']:<15} {n:<12} {bleu:<15.2f} {chrf:<15.2f}")
print("="*70)

# Сохранение результатов
with open('full_evaluation_results.txt', 'w', encoding='utf-8') as f:
    f.write("Full Test Set Evaluation Results\n")
    for lang, data in test_data.items():
        f.write(f"{data['name']}:\n")
        f.write(f"  Samples: {results[lang]['n_samples']}\n")
        f.write(f"  BLEU: {results[lang]['bleu']:.2f}\n")
        f.write(f"  chrF2: {results[lang]['chrf']:.2f}\n\n")

    f.write("Best translations per language:\n")
    for lang, data in test_data.items():
        f.write(f"\n{data['name']}:\n")
        for i in range(min(3, len(results[lang]['hyp']))):
            f.write(f"  {i+1}. SRC: {data['src'][i]}\n")
            f.write(f"     REF: {results[lang]['ref'][i]}\n")
            f.write(f"     HYP: {results[lang]['hyp'][i]}\n\n")


Оценка языка: Czech
Всего примеров: 2000, Батчей: 100


Czech: 100%|██████████| 100/100 [01:44<00:00,  1.05s/it]


Language: Czech | Total Samples: 2000
BLEU: 7.87 | chrF2: 22.05

Примеры переводов:

1. SRC: __cs__ - Drž hubu!
   REF: - Fuck you,
   HYP: - Shut up ! 

2. SRC: __cs__ - věrného a vzrušeného.
   REF: Loyal, and aroused.
   HYP: - A little girl and be crazy . 

3. SRC: __cs__ on již stane se v korporacním precinu století.
   REF: It has evolved to the corporate crime of the century.
   HYP: The most of the month on the German of the DD . 
Оценка языка: German
Всего примеров: 2000, Батчей: 100


German: 100%|██████████| 100/100 [02:10<00:00,  1.31s/it]


Language: German | Total Samples: 2000
BLEU: 8.33 | chrF2: 20.44

Примеры переводов:

1. SRC: __de__ 04:26:35
   REF: 04:26:35
   HYP: 04 : 26 : 26 

2. SRC: __de__ Prähistorische Archäologie im Dritten Reich".
   REF: Prähistorische Archäologie im Dritten Reich".
   HYP: The amount of the number of the amount of the amount of the new year . 

3. SRC: __de__ Die Nutzungsbedingungen werden durch das Klicken des Nutzers auf "Profil Speichern" vereinbart.
   REF: By clicking on 'Save profile', you the user agree to these terms and conditions.
   HYP: The Internet of the same system will be made by the same ' s work ' s work . 
Оценка языка: Danish
Всего примеров: 2000, Батчей: 100


Danish: 100%|██████████| 100/100 [02:01<00:00,  1.21s/it]


Language: Danish | Total Samples: 2000
BLEU: 7.64 | chrF2: 24.86

Примеры переводов:

1. SRC: __da__ under henvisning til Rådets forordning (EF) nr. 1405/2006 af 18. september 2006 om særlige foranstaltninger på landbrugsområdet til fordel for de mindre øer i Det Ægæiske Hav og om ændring af forordning (EF) nr. 1782/2003 [1], særlig artikel 14, og
   REF: Having regard to Council Regulation (EC) No 1405/2006 of 18 September 2006 laying down specific measures for agriculture in favour of the smaller Aegean islands and amending Regulation (EC) No 1782/2003 [1], and in particular Article 14 thereof,
   HYP: Having regard to Council Regulation ( EC ) No 1418 / 2006 of 18 September 2006 on the market in the market in the Republic of the Republic of the Republic of the market and in particular Article 14 ( 1 ) No 1314 / 2003 on the market and in particular Article 14 ( 1 ) thereof , 

2. SRC: __da__ Det bliver... mit mesterværk.
   REF: It's going to be my masterpiece.
   HYP: It ' s gonna b

Метрики, имеют существенные ограничения. Они оценивает совпадение с референсным переводом. Модель могла хорошо перевести, однако получить все равно низкую оценку.

Модель корректно переводит многие фразы и хорошо перключается между языками. Имеет проблемы с числами и датами.

**Ответьте своими словами в чем заключается техника back translation? Для чего она применяется и что позволяет получить? Опишите по шагам как ее применить к паре en->ru на данных из семинара. Сколько моделей понадобится? Сколько запусков обучения нужно будет сделать?**

backtranslation - это метод аугментации данных для машинного перевода, который позволяет использовать большие объемы одноязычных данных в целевом языке для улучшения качества модели перевода.
Например, перевод с навахо на английский. Параллельных данных мало, но английских текстов много. Можно использовать backtranslation для аугоментации.

Применяя к данным en->ru:
1. Есть большой корпус ru.
2. Обучаем обратную модель: ru → en.
3. Применяем модель ru → en к большому корпусу ru и получаем новые синтетические данные на en. Получаем пары, которые содержат реальные данные на русском и синтетические на английском. При генерации синтетических данных можно использовать разные параметры генерации.
4. Добавляем сгенерированное к исходным данным en->ru. Также можно сначала обучать на реальных данных, потом дообучать на синтетике или сэмплировать с весом.
5. Обучаем модель en->ru.

Таким образом, понадобится 2 модели, и используется  всего 2 запуска обучения.

Процесс можно повторять итеративно есть есть монолинвальные большие корпусы обоих языков и ресурсы.
